In [1]:
# Google Colab setup: fetch this repository and use this notebook's directory.
!pip install boto3
from pathlib import Path
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():
    !git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming

NOTEBOOK_DIR = REPO_ROOT / 'module_2/week_6/use_case_2'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 6.8 MB/s eta 0:00:00
Cloning into '/content/BITS_programming'...
remote: Enumerating objects: 2101, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 2101 (delta 11), reused 15 (delta 3), pack-reused 2066 (from 1)
Receiving objects: 100% (2101/2101), 263.06 MiB | 17.27 MiB/s, done.
Resolving deltas: 100% (400/400), done.
Updating files: 100% (1348/1348), done.
Working directory: /content/BITS_programming/module_2/week_6/use_case_2


In [2]:
# AWS credentials from Google Colab Secrets
# Makes boto3 / PySpark / AWS access work inside Colab.
import os

def get_colab_secret(name, required=True):
    try:
        from google.colab import userdata
        value = os.environ.get(name) or userdata.get(name)
    except Exception as exc:
        if required:
            raise RuntimeError(f"Unable to read Colab Secret: {name}") from exc
        return None
    if required and (value is None or value == ""):
        raise RuntimeError(f"Add the Colab Secret {name} and grant this notebook access.")
    return value

AWS_ACCESS_KEY_ID = get_colab_secret("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = get_colab_secret("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = get_colab_secret("AWS_SESSION_TOKEN", required=False)

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
if AWS_SESSION_TOKEN:
    os.environ["AWS_SESSION_TOKEN"] = AWS_SESSION_TOKEN

# Region: change this if your S3 buckets / Glue jobs live in another region.
_aws_region = os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION") or "ap-south-1"
os.environ["AWS_REGION"] = _aws_region
os.environ["AWS_DEFAULT_REGION"] = _aws_region

print(f"AWS credentials loaded. Region: {_aws_region}")

AWS credentials loaded. Region: ap-south-1


In [3]:
import os
import boto3
from botocore.exceptions import ClientError
try:
    from google.colab import userdata
except ImportError:
    userdata = None

# 1. Authenticate using Colab Secrets
os.environ["AWS_ACCESS_KEY_ID"] = os.environ.get("AWS_ACCESS_KEY_ID") or userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.environ.get("AWS_SECRET_ACCESS_KEY") or userdata.get("AWS_SECRET_ACCESS_KEY")
try:
    os.environ["AWS_SESSION_TOKEN"] = os.environ.get("AWS_SESSION_TOKEN") or userdata.get("AWS_SESSION_TOKEN")
except Exception:
    pass

# 2. Resolve Region and Account ID
region = os.environ.get("AWS_REGION", "ap-south-1")
sts_client = boto3.client("sts", region_name=region)
account_id = sts_client.get_caller_identity()["Account"]

_s3 = boto3.client("s3", region_name=region)

# 3. Dynamic bucket names with Account ID suffix
BASE_BUCKET_NAMES = ['usecase-etl-1', 'usecase-etl-2']
REQUIRED_BUCKETS = [f"{b}-{account_id}" for b in BASE_BUCKET_NAMES]

CREATED_BUCKETS = []

# 4. Create missing buckets idempotently with regional constraints
for _b in REQUIRED_BUCKETS:
    try:
        if region == "us-east-1":
            _s3.create_bucket(Bucket=_b)
        else:
            _s3.create_bucket(
                Bucket=_b,
                CreateBucketConfiguration={"LocationConstraint": region}
            )
        CREATED_BUCKETS.append(_b)
        print(f"Created bucket: {_b}")
    except ClientError as _e:
        _code = _e.response.get("Error", {}).get("Code", "")
        if _code in ("BucketAlreadyExists", "BucketAlreadyOwnedByYou", "Conflict"):
            print(f"Bucket already exists (reusing existing): {_b}")
        else:
            print(f"Could not create bucket {_b} ({_code}): {_e}")

print("Buckets ready:", REQUIRED_BUCKETS)

Bucket already exists (reusing existing): usecase-etl-1-455865672536
Bucket already exists (reusing existing): usecase-etl-2-455865672536
Buckets ready: ['usecase-etl-1-455865672536', 'usecase-etl-2-455865672536']


# Notebook 3B — AWS Glue PySpark ETL

**Purpose:** show how the same notebook logic from Use Case 2 becomes a repeatable **AWS Glue PySpark** pipeline.

## Where this notebook should be run
- **AWS Glue Studio notebook**, or
- **AWS Glue interactive session**


## Dependencies and AWS context

### Glue / Spark dependencies
- **SparkSession** — entry point for distributed processing
- **pyspark.sql.functions** — used for date parsing, null handling, derived columns, and aggregations

### One-time setup before the live Glue demo
Update the **bucket** and optional **prefix** in the config cell below.
Once updated, the notebook points to the exact S3 locations created by Use Case 1 and Use Case 2.


In [4]:
import os
import boto3
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
try:
    from google.colab import userdata
except ImportError:
    userdata = None

# 1. Load AWS credentials from Google Colab Secrets
os.environ["AWS_ACCESS_KEY_ID"] = os.environ.get("AWS_ACCESS_KEY_ID") or userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.environ.get("AWS_SECRET_ACCESS_KEY") or userdata.get("AWS_SECRET_ACCESS_KEY")
try:
    os.environ["AWS_SESSION_TOKEN"] = os.environ.get("AWS_SESSION_TOKEN") or userdata.get("AWS_SESSION_TOKEN")
except Exception:
    pass

# 2. Resolve AWS Region and Account ID dynamically
AWS_REGION = os.getenv('AWS_REGION', 'ap-south-1')
sts_client = boto3.client("sts", region_name=AWS_REGION)
account_id = sts_client.get_caller_identity()["Account"]

# 3. Dynamic bucket names
BUCKET_1 = f"usecase-etl-1-{account_id}"
BUCKET_2 = f"usecase-etl-2-{account_id}"

# 4. Define S3 Object Key and Local File Path
S3_KEY = "processed/retail_exploration_ready.csv"
LOCAL_INPUT_PATH = "/tmp/retail_exploration_ready.csv"

# 5. Download CSV directly from S3 using Boto3
s3_client = boto3.client("s3", region_name=AWS_REGION)
print(f"Downloading s3://{BUCKET_1}/{S3_KEY} to {LOCAL_INPUT_PATH}...")
s3_client.download_file(BUCKET_1, S3_KEY, LOCAL_INPUT_PATH)
print("Download complete!")

# 6. Initialize standard PySpark Session (No Hadoop S3 JARs needed)
if 'spark' in locals() and spark:
    spark.stop()

spark = SparkSession.builder \
    .appName('Glue-UseCase2-ETL') \
    .master('local[*]') \
    .getOrCreate()

# 7. Read Data from Local Disk
raw_df = (
    spark.read
         .option('header', 'true')
         .option('inferSchema', 'true')
         .csv(LOCAL_INPUT_PATH)
)

print('Raw row count:', raw_df.count())
raw_df.show(5, truncate=False)

Download complete!
Raw row count: 500
+---------+---------+---------------------------------+--------+----------------+---------+----------+--------------+-------------------+
|InvoiceNo|StockCode|Description                      |Quantity|InvoiceDate     |UnitPrice|CustomerID|Country       |InvoiceDateParsed  |
+---------+---------+---------------------------------+--------+----------------+---------+----------+--------------+-------------------+
|536365   |71053    |WHITE METAL LANTERN              |6       |02/01/2011 11:08|5.49     |17889.0   |Belgium       |2011-02-01 11:08:00|
|536365   |21730    |GLASS STAR FROSTED T-LIGHT HOLDER|2       |01/28/2011 11:32|4.22     |16943.0   |Germany       |2011-01-28 11:32:00|
|536366   |21730    |GLASS STAR FROSTED T-LIGHT HOLDER|6       |02/05/2011 08:48|5.8      |18065.0   |Netherlands   |2011-02-05 08:48:00|
|536366   |22752    |SET 7 BABUSHKA NESTING BOXES     |4       |01/13/2011 13:54|7.55     |14512.0   |United Kingdom|2011-01-13 13:54:

## Step 1 — Extract from S3

Glue reads the prepared file from a shared S3 location so every worker can access the same input.


In [5]:
raw_df = (
    spark.read
         .option('header', 'true')
         .option('inferSchema', 'true')
         .csv(LOCAL_INPUT_PATH)
)

print('Raw row count:', raw_df.count())
raw_df.show(5, truncate=False)


Raw row count: 500
+---------+---------+---------------------------------+--------+----------------+---------+----------+--------------+-------------------+
|InvoiceNo|StockCode|Description                      |Quantity|InvoiceDate     |UnitPrice|CustomerID|Country       |InvoiceDateParsed  |
+---------+---------+---------------------------------+--------+----------------+---------+----------+--------------+-------------------+
|536365   |71053    |WHITE METAL LANTERN              |6       |02/01/2011 11:08|5.49     |17889.0   |Belgium       |2011-02-01 11:08:00|
|536365   |21730    |GLASS STAR FROSTED T-LIGHT HOLDER|2       |01/28/2011 11:32|4.22     |16943.0   |Germany       |2011-01-28 11:32:00|
|536366   |21730    |GLASS STAR FROSTED T-LIGHT HOLDER|6       |02/05/2011 08:48|5.8      |18065.0   |Netherlands   |2011-02-05 08:48:00|
|536366   |22752    |SET 7 BABUSHKA NESTING BOXES     |4       |01/13/2011 13:54|7.55     |14512.0   |United Kingdom|2011-01-13 13:54:00|
|536366   |2173

## Step 2 — Transform and clean with PySpark

This cell mirrors the logic from Notebook 2 and Notebook 3, but in a distributed engine.


In [6]:
clean_df = (
    raw_df
      .withColumn('InvoiceDateTs', F.to_timestamp(F.col('InvoiceDate'), 'MM/dd/yyyy HH:mm'))
      .withColumn('Description', F.coalesce(F.col('Description'), F.lit('UNKNOWN_ITEM')))
      .withColumn('CustomerID', F.coalesce(F.col('CustomerID').cast('string'), F.lit('UNKNOWN_CUSTOMER')))
      .filter(F.col('InvoiceDateTs').isNotNull())
      .filter(F.col('UnitPrice') > 0)
      .withColumn('Revenue', F.col('Quantity') * F.col('UnitPrice'))
      .withColumn('IsReturn', F.col('Quantity') < 0)
      .withColumn('TransactionDate', F.to_date('InvoiceDateTs'))
      .withColumn('Month', F.date_format('InvoiceDateTs', 'yyyy-MM'))
      .withColumn('Category', F.split(F.col('Description'), ' ').getItem(0))
)

clean_df.show(5, truncate=False)


+---------+---------+---------------------------------+--------+----------------+---------+----------+--------------+-------------------+-------------------+-------+--------+---------------+-------+--------+
|InvoiceNo|StockCode|Description                      |Quantity|InvoiceDate     |UnitPrice|CustomerID|Country       |InvoiceDateParsed  |InvoiceDateTs      |Revenue|IsReturn|TransactionDate|Month  |Category|
+---------+---------+---------------------------------+--------+----------------+---------+----------+--------------+-------------------+-------------------+-------+--------+---------------+-------+--------+
|536365   |71053    |WHITE METAL LANTERN              |6       |02/01/2011 11:08|5.49     |17889.0   |Belgium       |2011-02-01 11:08:00|2011-02-01 11:08:00|32.94  |false   |2011-02-01     |2011-02|WHITE   |
|536365   |21730    |GLASS STAR FROSTED T-LIGHT HOLDER|2       |01/28/2011 11:32|4.22     |16943.0   |Germany       |2011-01-28 11:32:00|2011-01-28 11:32:00|8.44   |fal

## Step 3 — Aggregate into ETL outputs

This is the distributed equivalent of the pandas `groupby` logic.


In [7]:
daily_country_revenue = (
    clean_df.groupBy('TransactionDate', 'Country')
            .agg(F.round(F.sum('Revenue'), 2).alias('Revenue'))
)

monthly_category_revenue = (
    clean_df.groupBy('Month', 'Category')
            .agg(F.round(F.sum('Revenue'), 2).alias('Revenue'))
)

daily_country_revenue.show(10, truncate=False)
monthly_category_revenue.show(10, truncate=False)


+---------------+-----------+-------+
|TransactionDate|Country    |Revenue|
+---------------+-----------+-------+
|2011-02-23     |France     |9.52   |
|2011-02-20     |Belgium    |56.64  |
|2011-02-07     |Spain      |4.03   |
|2011-03-14     |Belgium    |7.12   |
|2011-02-08     |Spain      |32.76  |
|2011-03-30     |Germany    |72.36  |
|2011-02-03     |France     |85.71  |
|2011-01-17     |Netherlands|59.64  |
|2011-02-06     |France     |33.16  |
|2011-03-09     |Spain      |9.04   |
+---------------+-----------+-------+
only showing top 10 rows
+-------+------------+-------+
|Month  |Category    |Revenue|
+-------+------------+-------+
|2011-01|CREAM       |314.82 |
|2011-03|SET         |659.26 |
|2011-03|HAND        |1132.78|
|2011-02|HAND        |1298.91|
|2011-01|UNKNOWN_ITEM|232.68 |
|2011-03|CREAM       |377.02 |
|2011-02|GLASS       |619.36 |
|2011-02|ASSORTED    |472.16 |
|2011-03|UNKNOWN_ITEM|64.56  |
|2011-01|WHITE       |688.86 |
+-------+------------+-------+
only show

## Step 4 — Write outputs back to S3

Glue writes folders to S3 because Spark outputs distributed files rather than a single local CSV.


In [8]:
import os
import boto3

# 1. Define Local Paths and S3 Prefixes
LOCAL_OUT_DAILY = "/tmp/daily_country_revenue"
LOCAL_OUT_MONTHLY = "/tmp/monthly_category_revenue"

OUT_DAILY_PREFIX = "daily_country_revenue_glue"
OUT_MONTHLY_PREFIX = "monthly_category_revenue_glue"

# 2. Write DataFrames locally to disk
daily_country_revenue.write.mode('overwrite').option('header', 'true').csv(LOCAL_OUT_DAILY)
monthly_category_revenue.write.mode('overwrite').option('header', 'true').csv(LOCAL_OUT_MONTHLY)

# 3. Upload function for Spark output directories
def upload_spark_dir_to_s3(local_dir, bucket, s3_prefix):
    s3_client = boto3.client("s3", region_name=AWS_REGION)
    for root, _, files in os.walk(local_dir):
        for file in files:
            # Skip hidden metadata/checksum files (_SUCCESS, .crc)
            if file.startswith('.') or file.startswith('_'):
                continue
            local_file_path = os.path.join(root, file)
            s3_key = f"{s3_prefix}/{file}"
            s3_client.upload_file(local_file_path, bucket, s3_key)

# 4. Execute uploads
upload_spark_dir_to_s3(LOCAL_OUT_DAILY, BUCKET_2, OUT_DAILY_PREFIX)
upload_spark_dir_to_s3(LOCAL_OUT_MONTHLY, BUCKET_2, OUT_MONTHLY_PREFIX)

print('Glue ETL outputs written to:')
print(f"s3://{BUCKET_2}/{OUT_DAILY_PREFIX}/")
print(f"s3://{BUCKET_2}/{OUT_MONTHLY_PREFIX}/")

Glue ETL outputs written to:
s3://usecase-etl-2-455865672536/daily_country_revenue_glue/
s3://usecase-etl-2-455865672536/monthly_category_revenue_glue/


In [9]:
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-09-14 09:13:00
